# Pipeline Sift+Demons: detección de tejido, registro global y refinamiento local por tile

Se implementa el pipeline de registro de desarrollo propio **Sift+Demons** sobre dos fragmentos de tejido renal digitalizados de forma independiente (T1, tejido de referencia; T2, tejido móvil), ambos teñidos con tricrómico de Masson. 

El proceso comprende cuatro etapas secuenciales: 
1. Detección de tejido mediante GrandQC
2. Registro global rígido mediante SIFT mas RANSAC
3. Extracción de tiles bajo una grilla común con refinamiento local ECC/Demons
4. Exportación del conjunto de tiles pareados resultante.

## Configuración según la muestra a procesar

Este notebook procesa una muestra por ejecución. Para correrlo sobre Muestra1 o Muestra3, alcanza con modificar la ruta base del dataset a la carpeta correspondiente **en los dos bloques que la definen** -detección de tejido (GrandQC) y extracción de tiles-, antes de ejecutar el resto de las celdas:

```python
# Bloque GrandQC (detección de tejido)
ruta_dataset_imagenes = '/kaggle/input/datasets/candelapaez/muestra1nefro'   # o muestra3nefro

# Bloque de extracción de tiles
ruta_base_tif = '/kaggle/input/datasets/candelapaez/muestra1nefro'           # o muestra3nefro
```

Ambas variables deben apuntar siempre a la misma muestra. Los nombres de archivo (`muestra_1.tif`, `muestra_2.tif`) y el resto de la configuración no cambian entre muestras, ya que ambas siguen la misma convención T1 (referencia) / T2 (móvil).

## Detección de tejido con GrandQC

Se procede a la detección de tejido sobre cada fragmento (T1 y T2) de forma independiente, mediante la arquitectura UNet++ con encoder EfficientNetB0 provista por **GrandQC** (checkpoint `Tissue_Detection_MPP10.pth`, resolución de referencia
1,0 μm/px), que segmenta el tejido del fondo de vidrio y genera una máscara binaria por archivo. La resolución real de cada imagen (MPP) se obtiene de sus metadatos mediante `tiatoolbox`, empleando un valor de respaldo de 0,22 μm/px únicamente cuando no puede leerse directamente del archivo. El valor de MPP leído se registra en un archivo de metadatos (`metadata_mpp.json`) para su posterior auditoría y comparabilidad con los otros pipelines evaluados.

In [ ]:
# DETECCION DE TEJIDO CON MODELO GRANDQC - TISSUE DETECTION
import os
import shutil
import re
import json

print("MODELO GRAND QC APLICADO A NEFROLOGIA")

print("\nVerificando Archivos en Kaggle...")

# ruta_dataset_imagenes cambiar segun la muestra
ruta_dataset_imagenes = '/kaggle/input/datasets/candelapaez/muestra3nefro'  
drive_td = '/kaggle/input/datasets/candelapaez/modelograndqc/Tissue_Detection_MPP10.pth'
drive_qc = '/kaggle/input/datasets/candelapaez/modelograndqc/GrandQC_MPP15.pth'

archivo_tif_t1 = 'muestra_1.tif'
archivo_tif_t2 = 'muestra_2.tif'
ruta_tif_t1 = os.path.join(ruta_dataset_imagenes, archivo_tif_t1)
ruta_tif_t2 = os.path.join(ruta_dataset_imagenes, archivo_tif_t2)

if os.path.exists(ruta_dataset_imagenes):
    archivos_tif = [f for f in os.listdir(ruta_dataset_imagenes) if f.endswith('.tif')]
    if archivos_tif:
        print(f"Éxito! Se encontraron {len(archivos_tif)} imagen(es): {archivos_tif}")
    else:
        print("La carpeta existe, pero no hay archivos .tif adentro.")
else:
    print(f"ERROR: No se encontró la carpeta {ruta_dataset_imagenes}")

if os.path.exists(drive_td) and os.path.exists(drive_qc):
    print("Éxito! Se encontraron los archivos .pth del modelo.")
else:
    print("ERROR: No se encontraron los modelos .pth. Revisa las rutas.")

print("\nInstalando tiatoolbox para leer el MPP real de cada archivo...")
!pip install -q tiatoolbox

from tiatoolbox.wsicore.wsireader import WSIReader

VALOR_MPP_FALLBACK = 0.22  

def leer_mpp_real(ruta_tif, fallback=VALOR_MPP_FALLBACK):
    try:
        lector = WSIReader.open(ruta_tif)
        mpp = float(lector.info.mpp[0])
        print(f"  [{os.path.basename(ruta_tif)}] MPP leído del metadato: {mpp:.4f} µm/px")
        return mpp
    except Exception as e:
        print(f"  [{os.path.basename(ruta_tif)}] No se pudo leer el MPP real ({e}). "
              f"Se usa el valor de emergencia: {fallback}")
        return fallback

mpp_t1 = leer_mpp_real(ruta_tif_t1)
mpp_t2 = leer_mpp_real(ruta_tif_t2)

carpeta_resultados = '/kaggle/working/Resultados_GrandQC'
os.makedirs(carpeta_resultados, exist_ok=True)
ruta_metadata_mpp = os.path.join(carpeta_resultados, "metadata_mpp.json")
with open(ruta_metadata_mpp, "w") as f:
    json.dump({archivo_tif_t1: mpp_t1, archivo_tif_t2: mpp_t2}, f, indent=2)
print(f"\nMPP guardado para comparación: {ruta_metadata_mpp}")

# Preparacion del Entorno
%cd /kaggle/working
!rm -rf /kaggle/working/grandqc
!git clone https://github.com/cpath-ukk/grandqc.git /kaggle/working/grandqc

print("\n-> Instalando librerías requeridas...")
!apt-get update > /dev/null
!apt-get install -y openslide-tools > /dev/null
!pip install openslide-python segmentation_models_pytorch > /dev/null

base_path = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/models'
os.makedirs(f"{base_path}/td", exist_ok=True)
os.makedirs(f"{base_path}/qc", exist_ok=True)

local_td = f"{base_path}/td/Tissue_Detection_MPP10.pth"
local_qc = f"{base_path}/qc/GrandQC_MPP15.pth"

try:
    shutil.copy(drive_td, local_td)
    shutil.copy(drive_qc, local_qc)
    print("\nModelos vinculados correctamente a la carpeta de trabajo.")
except FileNotFoundError:
    print(f"\nERROR: No se pudieron copiar los modelos .pth.")

sh_path = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/run_tis.sh'
archivo_python = '/kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC/wsi_tis_detect.py'

with open(sh_path, 'r') as file:
    sh_content_original = file.read()
with open(archivo_python, 'r') as file:
    codigo_original = file.read()

tejidos_a_procesar = [
    (ruta_tif_t1, mpp_t1, '/kaggle/working/slide_temp_t1'),
    (ruta_tif_t2, mpp_t2, '/kaggle/working/slide_temp_t2'),
]

for ruta_tif_actual, mpp_actual, carpeta_temp in tejidos_a_procesar:
    os.makedirs(carpeta_temp, exist_ok=True)
    ruta_temp = os.path.join(carpeta_temp, os.path.basename(ruta_tif_actual))
    if not os.path.exists(ruta_temp):
        try:
            os.symlink(ruta_tif_actual, ruta_temp)
        except OSError:
            shutil.copy(ruta_tif_actual, ruta_temp)

    sh_mod = re.sub(r"^SLIDE_FOLDER=.*", f"SLIDE_FOLDER='{carpeta_temp}'", sh_content_original, flags=re.MULTILINE)
    sh_mod = re.sub(r"^OUTPUT_DIR=.*", f"OUTPUT_DIR='{carpeta_resultados}'", sh_mod, flags=re.MULTILINE)
    with open(sh_path, 'w') as f:
        f.write(sh_mod)

    # Parchear wsi_tis_detect.py con el MPP REAL de ESTE archivo
    patron_busqueda = r"slide\.properties\[['\"]openslide\.mpp-x['\"]\]"
    reemplazo = f"slide.properties.get('openslide.mpp-x', {mpp_actual})"
    codigo_mod = re.sub(patron_busqueda, reemplazo, codigo_original)
    silenciador = "import warnings\nwarnings.filterwarnings('ignore')\n"
    codigo_mod = silenciador + codigo_mod
    with open(archivo_python, 'w') as f:
        f.write(codigo_mod)

    print(f"\nCorriendo GrandQC tissue detection sobre {os.path.basename(ruta_tif_actual)} "
          f"(mpp={mpp_actual:.4f})...")
    %cd /kaggle/working/grandqc/01_WSI_inference_OPENSLIDE_QC
    !sh run_tis.sh

print("\nGrandQC finalizado para ambos tejidos.")
print(f"Máscaras generadas en: {carpeta_resultados}/tis_det_mask")

## Registro global rígido (SIFT + RANSAC)

Se ejecuta el registro global entre T1 y T2 a partir de las máscaras de tejido generadas en la etapa anterior. Cada fragmento se recorta a su bounding box de tejido a resolución macro y se normaliza mediante deconvolución óptica de
Beer-Lambert (recorte por percentiles 1-99 por canal), con el fin de reducir las diferencias de intensidad introducidas por la tinción. Sobre la imagen normalizada se aplica **CLAHE** (`clipLimit=3,0`, ventanas de 8×8) para realzar la textura celular, restringiendo la posterior detección de features **SIFT** (`nOctaveLayers=5`, `contrastThreshold=0,02`, `edgeThreshold=20`) a la región de tejido segmentada.

Los descriptores se emparejan mediante un matcher **FLANN** (índice KD-tree, 10 árboles) aplicando el test de ratio de Lowe (0,70, relajado a 0,80 si el resultado es insuficiente), y se filtran geométricamente descartando aquellos matches cuyo
desplazamiento se aparta más de un 20 % de la diagonal de la imagen respecto de la mediana del conjunto. Con los matches restantes se estima una transformación afín global mediante **RANSAC** (`estimateAffinePartial2D`, umbral adaptativo de al menos 5 px o 0,4 % de la diagonal, `maxIters=5000`, `confidence=0,995`), validada posteriormente por su escala y por el término homogéneo de la matriz resultante. La homografía obtenida a resolución macro se escala a resolución completa (nivel 0) considerando los factores de downsampling y los offsets de recorte propios de cada archivo, quedando disponible para la etapa de extracción de tiles.

In [ ]:
# REGISTRO GLOBAL RIGIDO SIFT + RANSAC
import cv2
import numpy as np
import matplotlib.pyplot as plt
import openslide
import os
import json
from PIL import Image

# ruta_base_tif cambiar segun la muestra 1 o 3 
ruta_base_tif = '/kaggle/input/datasets/candelapaez/muestra3nefro'
ruta_base_mascaras = '/kaggle/working/Resultados_GrandQC/tis_det_mask'

archivo_tif_t1 = 'muestra_1.tif'
archivo_tif_t2 = 'muestra_2.tif'

ruta_tif_t1 = os.path.join(ruta_base_tif, archivo_tif_t1)
ruta_tif_t2 = os.path.join(ruta_base_tif, archivo_tif_t2)
ruta_mascara_t1 = os.path.join(ruta_base_mascaras, archivo_tif_t1 + '_MASK.png')
ruta_mascara_t2 = os.path.join(ruta_base_mascaras, archivo_tif_t2 + '_MASK.png')


# offsets_recorte.json - comparabilidad entre pipelines.
ruta_offsets_json = os.path.join(ruta_base_tif, 'offsets_recorte.json')
with open(ruta_offsets_json, 'r') as f:
    offsets_recorte = json.load(f)

print("Contenido de offsets_recorte.json:", offsets_recorte)

offset_t1 = offsets_recorte.get(archivo_tif_t1, offsets_recorte.get('muestra_1', {'x': 0, 'y': 0}))
offset_t2 = offsets_recorte.get(archivo_tif_t2, offsets_recorte.get('muestra_2', {'x': 0, 'y': 0}))


# Normalizacion optica
def normalize_staining(img_rgb):
    img_f = np.clip(img_rgb.astype(np.float32) / 255.0, 1e-6, 1.0)
    od = -np.log(img_f)
    result = np.zeros_like(img_rgb)
    for ch in range(3):
        d = od[:, :, ch]
        lo, hi = np.percentile(d, [1, 99])
        norm = np.clip((d - lo) / (hi - lo + 1e-9), 0, 1)
        result[:, :, ch] = (norm * 255).astype(np.uint8)
    return result


# Resaltado de texturas
def preprocesar_para_sift(img_rgb):
    gray = (0.15 * img_rgb[:, :, 0] +
            0.70 * img_rgb[:, :, 1] +
            0.15 * img_rgb[:, :, 2]).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    return clahe.apply(gray)


# Mascra binaria excluyendo fondo blanco
def mascara_tejido_rgb(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(gray, 250, 255, cv2.THRESH_BINARY_INV)
    k = np.ones((3, 3), np.uint8)
    return cv2.erode(m, k, iterations=2)


# Validar H ≈ afín con escala ~1
def validar_homografia(H):
    if H is None:
        return False, "H es None"
    h22 = H[2, 2]
    if not (0.8 < h22 < 1.2):
        return False, f"H[2,2]={h22:.4f} fuera de [0.8, 1.2]"
    sx = np.sqrt(H[0, 0] ** 2 + H[1, 0] ** 2)
    sy = np.sqrt(H[0, 1] ** 2 + H[1, 1] ** 2)
    if abs(sx - 1.0) > 0.25 or abs(sy - 1.0) > 0.25:
        return False, f"Escala alejada de 1: sx={sx:.3f} sy={sy:.3f}"
    return True, f"ok  (H[2,2]={h22:.4f}, sx={sx:.3f}, sy={sy:.3f})"


# Filtro geométrico local de matches
def filtrar_matches_geometricos(kp1, kp2, matches, grid_size=8):
    if len(matches) < 4:
        return matches

    pts1 = np.float32([kp1[m.queryIdx].pt for m in matches])
    pts2 = np.float32([kp2[m.trainIdx].pt for m in matches])
    deltas = pts2 - pts1

    med_dx = np.median(deltas[:, 0])
    med_dy = np.median(deltas[:, 1])

    h = int(max(pts1[:, 1].max(), pts2[:, 1].max()))
    w = int(max(pts1[:, 0].max(), pts2[:, 0].max()))
    tol = np.sqrt(h ** 2 + w ** 2) * 0.20

    dist = np.sqrt((deltas[:, 0] - med_dx) ** 2 + (deltas[:, 1] - med_dy) ** 2)
    buenos_idx = np.where(dist < tol)[0]
    return [matches[i] for i in buenos_idx]


def bbox_global(gc):
    rects = [cv2.boundingRect(c) for c in gc]
    xn = min(x for x, y, w, h in rects)
    yn = min(y for x, y, w, h in rects)
    xx = max(x + w for x, y, w, h in rects)
    yx = max(y + h for x, y, w, h in rects)
    return xn, yn, xx - xn, yx - yn


# Escalado de H a nivel 0, con ds1/ds2 independientes por archivo
def escalar_H_a_nivel0(H_macro, ds1, ds2, bbox_t1_lvl0, bbox_t2_lvl0):
    S1 = np.array([[ds1, 0, 0],
                   [0, ds1, 0],
                   [0, 0, 1]], dtype=np.float64)
    S2_inv = np.array([[1 / ds2, 0, 0],
                        [0, 1 / ds2, 0],
                        [0, 0, 1]], dtype=np.float64)

    x1, y1 = bbox_t1_lvl0[:2]
    x2, y2 = bbox_t2_lvl0[:2]

    T1 = np.array([[1, 0, x1], [0, 1, y1], [0, 0, 1]], dtype=np.float64)
    T2_inv = np.array([[1, 0, -x2], [0, 1, -y2], [0, 0, 1]], dtype=np.float64)

    H_n0 = T1 @ S1 @ H_macro @ S2_inv @ T2_inv
    if abs(H_n0[2, 2]) < 1e-8:
        raise ValueError(
            "escalar_H_a_nivel0: H_n0[2,2] es ~0 (homografía degenerada). "
            "Revisar la cantidad de inliers de RANSAC antes de este paso."
        )
    H_n0 = H_n0 / H_n0[2, 2]
    return H_n0


def cargar_tejido_desde_slide(slide_path, mask_path, etiqueta="tejido"):
    slide = openslide.OpenSlide(slide_path)
    try:
        macro_level = min(3, slide.level_count - 1)
        macro_dims = slide.level_dimensions[macro_level]
        ds = slide.level_downsamples[macro_level]

        print(f"[{etiqueta}] Nivel macro: {macro_level}  (1 px macro = {ds:.2f} px en Nivel 0)")
        print(f"[{etiqueta}] Dimensiones nivel macro: {macro_dims}")

        macro_rgba = slide.read_region((0, 0), macro_level, macro_dims)
        macro_img = np.array(macro_rgba.convert('RGB'))

        qc_mask = np.array(Image.open(mask_path).convert('L'), dtype=np.uint8)
        if qc_mask.max() <= 1:
            qc_mask = (qc_mask * 255).astype(np.uint8)
        _, binary_mask_full = cv2.threshold(qc_mask, 127, 255, cv2.THRESH_BINARY)
        if binary_mask_full[0, 0] == 255:
            binary_mask_full = cv2.bitwise_not(binary_mask_full)

        k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        binary_mask_full = cv2.morphologyEx(binary_mask_full, cv2.MORPH_CLOSE, k_close)
        binary_mask_full = cv2.morphologyEx(binary_mask_full, cv2.MORPH_DILATE, k_close)

        w_macro, h_macro_px = macro_dims
        binary_mask_macro = np.array(
            Image.fromarray(binary_mask_full).resize((w_macro, h_macro_px), Image.NEAREST)
        )

        contours, _ = cv2.findContours(binary_mask_macro, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        valid_contours = [c for c in contours if cv2.contourArea(c) > 200]

        if not valid_contours:
            print(f"[{etiqueta}] No se detectó tejido en la máscara.")
            return None

        xm, ym, wm, hm = bbox_global(valid_contours)
        x0, y0, w0, h0 = int(xm * ds), int(ym * ds), int(wm * ds), int(hm * ds)

        crop_rgb = macro_img[ym:ym + hm, xm:xm + wm].copy()
        single_mask = np.zeros((h_macro_px, w_macro), dtype=np.uint8)
        cv2.drawContours(single_mask, valid_contours, -1, 255, -1)
        mask_crop = single_mask[ym:ym + hm, xm:xm + wm]

        tejido_clean = crop_rgb.copy()
        tejido_clean[mask_crop == 0] = 255

        return {
            'img_rgb': tejido_clean,
            'bbox_macro': (xm, ym, wm, hm),
            'bbox_nivel0': (x0, y0, w0, h0),
            'ds': ds,
        }
    finally:
        slide.close()


def alinear_tejidos_macro(slide_path_t1, mask_path_t1, slide_path_t2, mask_path_t2):
    print("\nALINEACION DE TEJIDOS...")
    print(f"T1: {os.path.basename(slide_path_t1)}  |  T2: {os.path.basename(slide_path_t2)}")

    t1_data = cargar_tejido_desde_slide(slide_path_t1, mask_path_t1, etiqueta="T1")
    t2_data = cargar_tejido_desde_slide(slide_path_t2, mask_path_t2, etiqueta="T2")
    if t1_data is None or t2_data is None:
        print("No se pudo extraer tejido de alguno de los dos archivos.")
        return None

    t1_rgb, t2_rgb = t1_data['img_rgb'], t2_data['img_rgb']

    print("\nPreprocesando para SIFT (normalización tinción + CLAHE)...")
    t1_norm = normalize_staining(t1_rgb)
    t2_norm = normalize_staining(t2_rgb)
    t1_gray = preprocesar_para_sift(t1_norm)
    t2_gray = preprocesar_para_sift(t2_norm)

    mask1 = mascara_tejido_rgb(t1_rgb)
    mask2 = mascara_tejido_rgb(t2_rgb)

    print("\nCalculando features SIFT sobre imágenes normalizadas...")
    sift = cv2.SIFT_create(nfeatures=0, nOctaveLayers=5, contrastThreshold=0.02, edgeThreshold=20)
    kp1, des1 = sift.detectAndCompute(t1_gray, mask1)
    kp2, des2 = sift.detectAndCompute(t2_gray, mask2)

    if des1 is None or des2 is None or len(kp1) < 4 or len(kp2) < 4:
        print("Sin suficientes features.")
        return None

    flann = cv2.FlannBasedMatcher({'algorithm': 1, 'trees': 10}, {'checks': 150})
    raw_matches = flann.knnMatch(des1, des2, k=2)

    buenos = [m for m, n in raw_matches if m.distance < 0.70 * n.distance]
    if len(buenos) < 10:
        buenos = [m for m, n in raw_matches if m.distance < 0.80 * n.distance]

    buenos = filtrar_matches_geometricos(kp1, kp2, buenos)
    print(f"  Matches post-filtro geométrico: {len(buenos)}")
    if len(buenos) < 4:
        print("Insuficientes matches tras el filtro.")
        return None

    src_pts = np.float32([kp1[m.queryIdx].pt for m in buenos]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in buenos]).reshape(-1, 1, 2)

    img_diag = np.sqrt(t1_gray.shape[0] ** 2 + t1_gray.shape[1] ** 2)
    ransac_thr = max(5.0, img_diag * 0.004)

    H_afin, mask_r = cv2.estimateAffinePartial2D(
        dst_pts, src_pts, method=cv2.RANSAC,
        ransacReprojThreshold=ransac_thr, maxIters=5000, confidence=0.995
    )
    inliers = int(mask_r.sum()) if mask_r is not None else 0
    print(f"  Inliers RANSAC Afín: {inliers} / {len(buenos)} (umbral={ransac_thr:.1f}px)")
    if H_afin is None:
        print("Transformación no calculada.")
        return None

    H_macro = np.vstack([H_afin, [0, 0, 1]])
    val_macro, msg_macro = validar_homografia(H_macro)
    print(f"  H_macro validación: {'✓' if val_macro else '⚠'} {msg_macro}")

    inlier_matches = [m for m, mk in zip(buenos, mask_r.ravel()) if mk]
    match_vis = cv2.drawMatches(
        t1_gray, kp1, t2_gray, kp2, inlier_matches, None,
        matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    plt.figure(figsize=(22, 7))
    plt.imshow(match_vis, cmap='gray')
    plt.title(f"Feature Matching — {inliers} inliers RANSAC  (H[2,2]={H_macro[2, 2]:.4f})")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    h1, w1 = t1_rgb.shape[:2]
    h2, w2 = t2_rgb.shape[:2]
    corners2 = np.float32([[0, 0], [w2, 0], [w2, h2], [0, h2]]).reshape(-1, 1, 2)
    corners2_t = cv2.perspectiveTransform(corners2, H_macro)
    all_c = np.concatenate([
        np.float32([[0, 0], [w1, 0], [w1, h1], [0, h1]]).reshape(-1, 1, 2),
        corners2_t
    ])
    off_x = max(0, int(-np.floor(all_c[:, :, 0].min())))
    off_y = max(0, int(-np.floor(all_c[:, :, 1].min())))
    canvas_w = int(np.ceil(all_c[:, :, 0].max())) + off_x
    canvas_h = int(np.ceil(all_c[:, :, 1].max())) + off_y

    T_off = np.array([[1, 0, off_x], [0, 1, off_y], [0, 0, 1]], dtype=np.float64)
    H_off = T_off @ H_macro

    canvas1 = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * 255
    canvas1[off_y:off_y + h1, off_x:off_x + w1] = t1_rgb
    canvas2_w = cv2.warpPerspective(t2_rgb, H_off, (canvas_w, canvas_h), borderValue=(255, 255, 255))

    m1 = mascara_tejido_rgb(canvas1)
    m2 = mascara_tejido_rgb(canvas2_w)
    overlap = cv2.bitwise_and(m1, m2)
    solo_t2 = cv2.bitwise_and(m2, cv2.bitwise_not(m1))

    super_img = canvas1.copy()
    super_img[overlap > 0] = (
        canvas1[overlap > 0].astype(np.float32) * 0.5 +
        canvas2_w[overlap > 0].astype(np.float32) * 0.5
    ).astype(np.uint8)
    super_img[solo_t2 > 0] = canvas2_w[solo_t2 > 0]

    area_t1 = np.sum(m1 > 0)
    area_overlap = np.sum(overlap > 0)
    pct_overlap = 100 * area_overlap / area_t1 if area_t1 > 0 else 0

    fig, axes = plt.subplots(1, 3, figsize=(24, 8))
    axes[0].imshow(t1_rgb); axes[0].set_title("Tejido 1 (Referencia)"); axes[0].axis('off')
    axes[1].imshow(t2_rgb); axes[1].set_title("Tejido 2 (Original)"); axes[1].axis('off')
    axes[2].imshow(super_img); axes[2].set_title(f"Superposición alineada\n{inliers} inliers | {pct_overlap:.0f}% overlap"); axes[2].axis('off')
    plt.tight_layout()
    plt.show()

    H_nivel0 = escalar_H_a_nivel0(H_macro, t1_data['ds'], t2_data['ds'],
                                   t1_data['bbox_nivel0'], t2_data['bbox_nivel0'])

    return {
        'H_macro': H_macro,
        'H_nivel0': H_nivel0,
        'inliers': inliers,
        'overlap_pct': pct_overlap,
        'tejido_1': {'origen_lvl0': t1_data['bbox_nivel0'][:2], 'dims_lvl0': t1_data['bbox_nivel0'][2:]},
        'tejido_2': {'origen_lvl0': t2_data['bbox_nivel0'][:2], 'dims_lvl0': t2_data['bbox_nivel0'][2:]},
    }


# Ejecucion
if all(os.path.exists(p) for p in [ruta_tif_t1, ruta_mascara_t1, ruta_tif_t2, ruta_mascara_t2]):
    datos = alinear_tejidos_macro(ruta_tif_t1, ruta_mascara_t1, ruta_tif_t2, ruta_mascara_t2)
    if datos:
        print("\nRESUMEN FINAL")
        print(f"Tejido 1: origen={datos['tejido_1']['origen_lvl0']}  tamaño={datos['tejido_1']['dims_lvl0']}")
        print(f"Tejido 2: origen={datos['tejido_2']['origen_lvl0']}  tamaño={datos['tejido_2']['dims_lvl0']}")
        print(f"Inliers RANSAC: {datos['inliers']}")
        print(f"Overlap: {datos['overlap_pct']:.1f}%")
else:
    print("Archivos no encontrados. Verificá las rutas.")

## Extracción de tiles y refinamiento local (ECC + Demons)

Sobre la transformación global escalada a resolución completa, se define una grilla común de extracción (tamaño de tile de 512 px, stride de 256 px, 50 % de solapamiento), descartando los tiles cuyo porcentaje de tejido sea inferior al
40 % o cuya informatividad -estimada mediante la varianza del Laplaciano- resulte menor al umbral establecido. Cada tile se extrae con un margen de contexto adicional de 128 px por lado, empleado únicamente durante el refinamiento local y
recortado antes del guardado final.

Sobre cada par de tiles se aplica un refinamiento local en dos etapas: primero un ajuste sub-píxel mediante **ECC** (`MOTION_EUCLIDEAN`, 50 iteraciones, tolerancia 1e-4) y, únicamente si este converge, una deformación no rígida mediante un filtro **Demons** simétrico de SimpleITK (`SymmetricForcesDemonsRegistrationFilter`, 25 iteraciones, suavizado gaussiano de 2,5, con el campo de desplazamiento acotado a ±25 px). 

Los tiles cuyo ajuste ECC no converge conservan únicamente el resultado de la alineación afín global y se clasifican, a fines diagnósticos, como "geom" en lugar de "ecc". Adicionalmente, se calcula por tile una métrica de confianza de alineación interna mediante correlación de fase sobre los gradientes de intensidad de ambos tiles, almacenada como metadato de contexto para su comparación posterior con las métricas externas de calidad.

In [ ]:
# EXTRACCION DE TILES ECC + DEMONS 
import cv2
import numpy as np
import matplotlib.pyplot as plt
import openslide
import os
import itertools
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

try:
    import SimpleITK as sitk
    SITK_DISPONIBLE = True
except ImportError:
    SITK_DISPONIBLE = False
    print("AVISO: SimpleITK no está instalado (pip install SimpleITK).")
    print("       La deformación no rígida se omitirá; el resto del pipeline funciona igual.")

# ruta_base_tif cambiar segun la muestra 1 o 3 
ruta_base_tif = '/kaggle/input/datasets/candelapaez/muestra3nefro'
ruta_base_mascaras = '/kaggle/working/Resultados_GrandQC/tis_det_mask'

archivo_tif_t1 = 'muestra_1.tif'
archivo_tif_t2 = 'muestra_2.tif'

ruta_tif_t1 = os.path.join(ruta_base_tif, archivo_tif_t1)
ruta_tif_t2 = os.path.join(ruta_base_tif, archivo_tif_t2)
ruta_mascara_t1 = os.path.join(ruta_base_mascaras, archivo_tif_t1 + '_MASK.png')
ruta_mascara_t2 = os.path.join(ruta_base_mascaras, archivo_tif_t2 + '_MASK.png')

carpeta_base = '/kaggle/working/tiles_apareados_aumentados'

carpeta_ecc_t1 = f'{carpeta_base}/ecc/tejido1'
carpeta_ecc_t2 = f'{carpeta_base}/ecc/tejido2'
carpeta_geom_t1 = f'{carpeta_base}/geom/tejido1'
carpeta_geom_t2 = f'{carpeta_base}/geom/tejido2'

for carpeta in [carpeta_ecc_t1, carpeta_ecc_t2, carpeta_geom_t1, carpeta_geom_t2]:
    os.makedirs(carpeta, exist_ok=True)

# Parámetros
TILE_SIZE = 512
STRIDE = 256           
MIN_TEJIDO_PCT = 0.40

MIN_INFORMATIVIDAD = 15.0

PAD = 128
WORK_SIZE = TILE_SIZE + 2 * PAD

DEFORMACION_ITERACIONES = 25
DEFORMACION_SUAVIZADO = 2.5
DEFORMACION_MAX_DESPLAZ = 25


# Máscara binaria excluyendo fondo blanco.
def mascara_tejido_rgb(img_rgb):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    _, m = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
    k = np.ones((5, 5), np.uint8)
    return cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)


# Mide cuánta "estructura" (bordes, textura) tiene un tile.
def medir_informatividad(img_rgb, mask=None):
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    if mask is not None and np.sum(mask > 0) > 0:
        valores = gray[mask > 0]
        if valores.size < 100:
            return 0.0
        lap = cv2.Laplacian(gray, cv2.CV_64F)
        return float(lap[mask > 0].var())
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    return float(lap.var())


# Métrica de confianza de alineación por tile
_clahe_confianza = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))


def medir_confianza_alineacion(img1_rgb, img2_rgb):
    g1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
    g2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)
    g1 = _clahe_confianza.apply(g1)
    g2 = _clahe_confianza.apply(g2)

    gx1 = cv2.Sobel(g1, cv2.CV_32F, 1, 0, ksize=3)
    gy1 = cv2.Sobel(g1, cv2.CV_32F, 0, 1, ksize=3)
    f1 = cv2.magnitude(gx1, gy1)

    gx2 = cv2.Sobel(g2, cv2.CV_32F, 1, 0, ksize=3)
    gy2 = cv2.Sobel(g2, cv2.CV_32F, 0, 1, ksize=3)
    f2 = cv2.magnitude(gx2, gy2)

    win = cv2.createHanningWindow((f1.shape[1], f1.shape[0]), cv2.CV_32F)
    (dx, dy), response = cv2.phaseCorrelate(f1, f2, win)
    return {
        "shift_residual_px": float(np.hypot(dx, dy)),
        "confianza_alineacion": float(response),
    }


# Micro-ajuste sub-píxel con MOTION_EUCLIDEAN
def micro_alineacion_ecc(img1_rgb, img2_rgb):
    gray1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)

    warp_mode = cv2.MOTION_EUCLIDEAN
    warp_matrix = np.eye(2, 3, dtype=np.float32)

    iteraciones = 50
    tolerancia = 1e-4
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, iteraciones, tolerancia)

    try:
        _, warp_matrix = cv2.findTransformECC(gray1, gray2, warp_matrix, warp_mode, criteria)
        img2_alineada = cv2.warpAffine(
            img2_rgb, warp_matrix, (img1_rgb.shape[1], img1_rgb.shape[0]),
            flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
            borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255)
        )
        return img2_alineada, True

    except Exception:
        return img2_rgb, False


# Refinamiento elástico (no rígido) con Demons/SimpleITK
def deformacion_no_rigida(img1_rgb, img2_rgb,
                           iteraciones=DEFORMACION_ITERACIONES,
                           suavizado=DEFORMACION_SUAVIZADO,
                           max_desplazamiento=DEFORMACION_MAX_DESPLAZ):
    if not SITK_DISPONIBLE:
        return img2_rgb, False

    gray1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    gray2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)

    try:
        fixed = sitk.GetImageFromArray(gray1)
        moving = sitk.GetImageFromArray(gray2)

        demons = sitk.SymmetricForcesDemonsRegistrationFilter()
        demons.SetNumberOfIterations(iteraciones)
        demons.SetStandardDeviations(suavizado)

        campo = demons.Execute(fixed, moving)
        disp = sitk.GetArrayFromImage(campo)

    except Exception:
        return img2_rgb, False

    disp = np.clip(disp, -max_desplazamiento, max_desplazamiento)

    h, w = gray1.shape
    xs_grid, ys_grid = np.meshgrid(np.arange(w), np.arange(h))
    map_x = (xs_grid + disp[..., 0]).astype(np.float32)
    map_y = (ys_grid + disp[..., 1]).astype(np.float32)

    img2_deformada = cv2.remap(
        img2_rgb, map_x, map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT, borderValue=(255, 255, 255)
    )
    return img2_deformada, True



def extraer_tiles_micro(slide_path_t1, mask_path_t1, slide_path_t2, datos_alineacion, offset_t1, offset_t2):
    print("\nEXTRACCIÓN DE TILES")
    if SITK_DISPONIBLE:
        print(f"Deformación no rígida (Demons): ACTIVADA "
              f"(iter={DEFORMACION_ITERACIONES}, suavizado={DEFORMACION_SUAVIZADO}, "
              f"max_desplaz={DEFORMACION_MAX_DESPLAZ}px)")
    else:
        print("Deformación no rígida: DESACTIVADA (SimpleITK no instalado)")
    print(f"Filtro de tejido mínimo: {MIN_TEJIDO_PCT*100:.0f}%")
    print(f"Filtro de informatividad (varianza Laplaciano): >= {MIN_INFORMATIVIDAD}")

    H_nivel0 = datos_alineacion['H_nivel0']

    try:
        H_inv = np.linalg.inv(H_nivel0)
    except np.linalg.LinAlgError as e:
        raise ValueError(
            "extraer_tiles_micro: H_nivel0 es singular (no invertible). "
            "Revisar los inliers de RANSAC / la validación de la homografía "
            "en el bloque de alineación antes de extraer tiles."
        ) from e

    t1_orig = datos_alineacion['tejido_1']['origen_lvl0']
    t1_dims = datos_alineacion['tejido_1']['dims_lvl0']

    xs = range(t1_orig[0], t1_orig[0] + t1_dims[0] - TILE_SIZE, STRIDE)
    ys = range(t1_orig[1], t1_orig[1] + t1_dims[1] - TILE_SIZE, STRIDE)
    total_candidatos = len(xs) * len(ys)

    tiles_ecc = 0
    tiles_geom = 0
    tiles_deformados = 0
    tiles_descartados = 0
    tiles_descartados_tejido = 0
    tiles_descartados_informatividad = 0
    tiles_descartados_lectura = 0
    pares_validos = []

    print(f"Analizando grilla: {len(xs)} columnas x {len(ys)} filas "
          f"({total_candidatos} candidatos totales)...")
    print(f"Contexto extra por lado: {PAD}px  (canvas de trabajo: {WORK_SIZE}x{WORK_SIZE})")

    slide1 = openslide.OpenSlide(slide_path_t1)
    slide2 = openslide.OpenSlide(slide_path_t2)
    try:
        nivel0_dims_t1 = slide1.level_dimensions[0]
        nivel0_dims_t2 = slide2.level_dimensions[0]  

        # Escalar máscara GrandQC de T1 al nivel 0 de T1
        qc_mask_pil = Image.open(mask_path_t1).convert('L')
        qc_mask_full = np.array(qc_mask_pil.resize(nivel0_dims_t1, Image.NEAREST), dtype=np.uint8)
        if qc_mask_full.max() <= 1:
            qc_mask_full = (qc_mask_full * 255).astype(np.uint8)
        _, qc_binary = cv2.threshold(qc_mask_full, 127, 255, cv2.THRESH_BINARY)
        if qc_binary[0, 0] == 255:
            qc_binary = cv2.bitwise_not(qc_binary)

        pbar = tqdm(
            itertools.product(ys, xs),
            total=total_candidatos,
            desc="Extrayendo tiles",
            unit="tile",
        )

        for y1, x1 in pbar:
            # Filtro de tejido en T1
            roi_qc = qc_binary[y1:y1+TILE_SIZE, x1:x1+TILE_SIZE]
            if roi_qc.size == 0 or (np.sum(roi_qc > 0) / (TILE_SIZE**2) < MIN_TEJIDO_PCT):
                tiles_descartados += 1
                tiles_descartados_tejido += 1
                pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                                  geom=tiles_geom, descartados=tiles_descartados)
                continue

            xw1, yw1 = x1 - PAD, y1 - PAD
            xw2, yw2 = x1 + TILE_SIZE + PAD, y1 + TILE_SIZE + PAD

            if xw1 < 0 or yw1 < 0 or xw2 > nivel0_dims_t1[0] or yw2 > nivel0_dims_t1[1]:
                tiles_descartados += 1
                pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                                  geom=tiles_geom, descartados=tiles_descartados)
                continue
            try:
                t1_work_rgba = slide1.read_region((xw1, yw1), 0, (WORK_SIZE, WORK_SIZE))
                t1_work_rgb = np.array(t1_work_rgba.convert('RGB'))
            except Exception:
                tiles_descartados += 1
                tiles_descartados_lectura += 1
                continue

            esquinas_t1 = np.float32([
                [xw1, yw1],
                [xw1+WORK_SIZE, yw1],
                [xw1+WORK_SIZE, yw1+WORK_SIZE],
                [xw1, yw1+WORK_SIZE]
            ]).reshape(-1, 1, 2)
            esquinas_t2 = cv2.perspectiveTransform(esquinas_t1, H_inv)

            x2_min = max(0, int(np.floor(esquinas_t2[:, 0, 0].min())))
            y2_min = max(0, int(np.floor(esquinas_t2[:, 0, 1].min())))
            x2_max = min(nivel0_dims_t2[0], int(np.ceil(esquinas_t2[:, 0, 0].max())))
            y2_max = min(nivel0_dims_t2[1], int(np.ceil(esquinas_t2[:, 0, 1].max())))
            w2, h2 = x2_max - x2_min, y2_max - y2_min

            if w2 <= 0 or h2 <= 0 or w2 > WORK_SIZE * 2 or h2 > WORK_SIZE * 2:
                tiles_descartados += 1
                pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                                  geom=tiles_geom, descartados=tiles_descartados)
                continue
            try:
                t2_raw_rgba = slide2.read_region((x2_min, y2_min), 0, (w2, h2))
                t2_raw_rgb = np.array(t2_raw_rgba.convert('RGB'))
            except Exception:
                tiles_descartados += 1
                tiles_descartados_lectura += 1
                continue

            T_tile1 = np.array([[1, 0, -xw1], [0, 1, -yw1], [0, 0, 1]], dtype=np.float64)
            T_tile2inv = np.array([[1, 0, x2_min], [0, 1, y2_min], [0, 0, 1]], dtype=np.float64)
            H_local = T_tile1 @ H_nivel0 @ T_tile2inv

            t2_geom_work = cv2.warpPerspective(
                t2_raw_rgb, H_local, (WORK_SIZE, WORK_SIZE),
                borderValue=(255, 255, 255)
            )

            t2_work, ecc_exitoso = micro_alineacion_ecc(t1_work_rgb, t2_geom_work)

            if ecc_exitoso:
                t2_final_work, deform_exitoso = deformacion_no_rigida(t1_work_rgb, t2_work)
            else:
                t2_final_work = t2_work
                deform_exitoso = False

            t1_rgb = t1_work_rgb[PAD:PAD+TILE_SIZE, PAD:PAD+TILE_SIZE]
            t2_final = t2_final_work[PAD:PAD+TILE_SIZE, PAD:PAD+TILE_SIZE]

            mask_t1 = mascara_tejido_rgb(t1_rgb)
            mask_t2 = mascara_tejido_rgb(t2_final)
            if np.sum(mask_t2 > 0) / (TILE_SIZE**2) < MIN_TEJIDO_PCT:
                tiles_descartados += 1
                tiles_descartados_tejido += 1
                pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                                  geom=tiles_geom, descartados=tiles_descartados)
                continue

            info_t1 = medir_informatividad(t1_rgb, mask_t1)
            info_t2 = medir_informatividad(t2_final, mask_t2)
            if info_t1 < MIN_INFORMATIVIDAD or info_t2 < MIN_INFORMATIVIDAD:
                tiles_descartados += 1
                tiles_descartados_informatividad += 1
                pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                                  geom=tiles_geom, descartados=tiles_descartados)
                continue

            confianza = medir_confianza_alineacion(t1_rgb, t2_final)

            # Coordenadas ABSOLUTAS (offset del recorte + posición local)
            x1_orig = offset_t1.get('x', 0) + x1
            y1_orig = offset_t1.get('y', 0) + y1
            x2_orig = offset_t2.get('x', 0) + x2_min
            y2_orig = offset_t2.get('y', 0) + y2_min

            idx = tiles_ecc + tiles_geom
            nombre = f"tile_{idx:05d}_r{y1}_c{x1}.png"

            if ecc_exitoso:
                Image.fromarray(t1_rgb).save(f"{carpeta_ecc_t1}/{nombre}")
                Image.fromarray(t2_final).save(f"{carpeta_ecc_t2}/{nombre}")
                tiles_ecc += 1
                if deform_exitoso:
                    tiles_deformados += 1
            else:
                Image.fromarray(t1_rgb).save(f"{carpeta_geom_t1}/{nombre}")
                Image.fromarray(t2_final).save(f"{carpeta_geom_t2}/{nombre}")
                tiles_geom += 1

            pares_validos.append({
                'nombre': nombre,
                'carpeta': 'ecc' if ecc_exitoso else 'geom',
                'x1': x1, 'y1': y1,
                'ecc_exitoso': ecc_exitoso,
                'deform_exitoso': deform_exitoso,
                'informatividad_t1': round(info_t1, 2),
                'informatividad_t2': round(info_t2, 2),
                'shift_residual_px': round(confianza['shift_residual_px'], 2),
                'confianza_alineacion': round(confianza['confianza_alineacion'], 4),
                'bbox_t1_lvl0': (x1, y1, TILE_SIZE, TILE_SIZE),
                'bbox_t2_lvl0': (x2_min, y2_min, w2, h2),
                'x1_orig': x1_orig, 'y1_orig': y1_orig,   
                'x2_orig': x2_orig, 'y2_orig': y2_orig,
            })

            pbar.set_postfix(guardados=tiles_ecc + tiles_geom, ecc=tiles_ecc,
                              geom=tiles_geom, descartados=tiles_descartados)

        pbar.close()

    finally:
        slide1.close()
        slide2.close()

    total_guardados = tiles_ecc + tiles_geom
    total_analizados = total_guardados + tiles_descartados
    pct_exito = (total_guardados / total_analizados * 100) if total_analizados > 0 else 0
    otros_descartados = (tiles_descartados - tiles_descartados_tejido
                          - tiles_descartados_informatividad - tiles_descartados_lectura)

    print(f"\nEXTRACCIÓN FINALIZADA!")
    print(f"Resumen ({total_analizados} recuadros analizados):")
    print(f" -> ecc/   (alineación fina completa, incluye las que además tienen deformación): {tiles_ecc} tiles")
    print(f"      de las cuales con refinamiento no rígido aplicado: {tiles_deformados} tiles")
    print(f" -> geom/  (solo alineación geométrica, ECC no convergió): {tiles_geom} tiles")
    print(f" -> Total guardados: {total_guardados} ({pct_exito:.1f}% del área)")
    print(f" -> Descartados: {tiles_descartados}")
    print(f"      · por tejido insuficiente (<{MIN_TEJIDO_PCT*100:.0f}%): {tiles_descartados_tejido}")
    print(f"      · por baja informatividad (<{MIN_INFORMATIVIDAD}): {tiles_descartados_informatividad}")
    print(f"      · por error de lectura del TIF (openslide/IO): {tiles_descartados_lectura}")
    print(f"      · otros (borde/sin contexto/sin encaje): {otros_descartados}")
    print(f"\nRutas:")
    print(f"  {carpeta_base}/ecc/tejido1  |  ecc/tejido2")
    print(f"  {carpeta_base}/geom/tejido1 |  geom/tejido2")

    manifest_path = f"{carpeta_base}/manifest.csv"
    df_manifest = pd.DataFrame(pares_validos)
    df_manifest.to_csv(manifest_path, index=False)
    print(f" -> Manifest guardado en: {manifest_path}")

    if len(df_manifest) > 0 and 'confianza_alineacion' in df_manifest.columns:
        print(f"\nConfianza de alineación por tile (phase correlation sobre gradiente):")
        print(f"  Mediana: {df_manifest['confianza_alineacion'].median():.3f}  "
              f"| Percentil 10 (peor 10%): {df_manifest['confianza_alineacion'].quantile(0.10):.3f}")
        peores_5 = df_manifest.nsmallest(5, 'confianza_alineacion')[
            ['nombre', 'confianza_alineacion', 'shift_residual_px']
        ]
        print("  5 tiles con menor confianza (candidatos a revisar a mano):")
        print(peores_5.to_string(index=False))

    return pares_validos


def visualizar_resultados(pares, n_muestras=4):
    if not pares:
        print("No hay pares para mostrar.")
        return

    import random
    muestra = random.sample(pares, min(n_muestras, len(pares)))


    fig, axes = plt.subplots(len(muestra), 3, figsize=(15, 5 * len(muestra)))
    if len(muestra) == 1:
        axes = [axes]

    for i, par in enumerate(muestra):
        nombre = par['nombre']
        tipo = "ECC" if par['ecc_exitoso'] else "GEOM"
        if par.get('deform_exitoso'):
            tipo += "+DEF"
        t1_path = f"{carpeta_ecc_t1 if par['ecc_exitoso'] else carpeta_geom_t1}/{nombre}"
        t2_path = f"{carpeta_ecc_t2 if par['ecc_exitoso'] else carpeta_geom_t2}/{nombre}"

        t1 = np.array(Image.open(t1_path))
        t2 = np.array(Image.open(t2_path))
        superpuesto = cv2.addWeighted(t1, 0.5, t2, 0.5, 0)
        
        axes[i][0].imshow(t1); axes[i][0].set_title(f"T1 (Ref)\n{nombre}\ninfo={par['informatividad_t1']:.0f}"); axes[i][0].axis('off')
        axes[i][1].imshow(t2); axes[i][1].set_title(f"T2 ({tipo})\ninfo={par['informatividad_t2']:.0f}"); axes[i][1].axis('off')
        axes[i][2].imshow(superpuesto); axes[i][2].set_title("Superposición"); axes[i][2].axis('off')

    plt.tight_layout()
    plt.show()


# Ejecucion
if 'datos' in locals() and datos is not None:
    pares = extraer_tiles_micro(ruta_tif_t1, ruta_mascara_t1, ruta_tif_t2, datos, offset_t1, offset_t2)

    print("\nVISUALIZACIÓN DE RESULTADOS")
    cantidad_mostrar = 10
    print(f"\nGenerando visualización de {cantidad_mostrar} muestras aleatorias...")
    visualizar_resultados(pares, n_muestras=cantidad_mostrar)

else:
    print("Falta la variable 'datos'. Ejecuta el bloque de alineación primero!")

## Exportación del conjunto de tiles pareados

Finalmente, se comprime el conjunto completo de tiles pareados generado -incluyendo los subconjuntos clasificados como "ecc" y "geom" para ambos tejidos- en un archivo `.zip`, quedando disponible para su descarga y su posterior incorporación al cálculo de métricas de calidad de alineación descripto en la sección de Métricas.

La carpeta resultante se organiza según el criterio de convergencia del refinamiento ECC y el tejido correspondiente:

```
tiles_apareados_aumentados/
├── ecc/                    ← tiles con ajuste ECC convergente (+ deformación Demons)
│   ├── tejido1/             (tiles de T1, referencia)
│   └── tejido2/             (tiles de T2, alineados)
└── geom/                   ← tiles con ECC no convergente (solo alineación afín global)
    ├── tejido1/
    └── tejido2/
```

In [ ]:
# GUARDADO DE TILES EN CARPETA .ZIP
import shutil
import os
from IPython.display import FileLink, display


# Compresión y exportación del dataset
os.chdir('/kaggle/working')

carpeta_tiles = '/kaggle/working/tiles_apareados_aumentados'
nombre_zip_base = '/kaggle/working/TilesSiftDemonsMuestra3'

print(f"\nComprimiendo la carpeta: {carpeta_tiles}...")

# Comprimir la carpeta entera en formato ZIP
shutil.make_archive(nombre_zip_base, 'zip', carpeta_tiles)

archivo_generado = f"{nombre_zip_base}.zip"
peso_mb = os.path.getsize(archivo_generado) / (1024 * 1024)


print(f"\nCompresión finalizada!")
print(f"Archivo: {archivo_generado}")
print(f"Peso aproximado: {peso_mb:.2f} MB")

# Generar un link para descargar 
display(FileLink('TilesSiftDemonsMuestra3.zip'))